# **MÓDULO 2: LIMPIEZA Y ESTANDARIZACIÓN DEL DATA LAKE**
## IONClinics & Universidad Complutense de Madrid

## **Pre-Work Checklist: Ejecutar antes de comenzar**

Esta etapa inicial prepara el entorno de trabajo para la ejecución del sistema. Aquí se cargan las librerías necesarias, se configuran las rutas a las bases de datos y archivos requeridos, y se establecen las sesiones de Spark. Este paso es fundamental para garantizar que todos los procesos posteriores se ejecuten de manera fluida y sin errores relacionados con la configuración o dependencias del entorno.

In [ ]:
from google.colab import drive
import os, sys
drive.mount('/content/drive')
print(os.getcwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1E1kpR8gbMGkDTzr9NwttHfsFgd8D-cjG/Data Lake IONClinics


In [ ]:
os.chdir('/content/drive/MyDrive/Data Lake IONClinics')

In [ ]:
from configparser import ConfigParser
from pathlib import Path
import requests
import urllib
import json
import pandas as pd
import numpy as np
import random

In [ ]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install --upgrade pyspark
!pip install py4j
!pip install --upgrade openai

import sys
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.sql.functions import col, explode, from_json, substring, split, udf,lower, expr, regexp_replace, monotonically_increasing_id
from pyspark.sql.types import StringType, StructType, StructField, IntegerType, ArrayType
from pyspark.sql.utils import AnalysisException
from pyspark.sql import Row
from pyspark.sql import Window

import openai
from openai import OpenAI
import re
import ast


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,927 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,211 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages 

### **Parameter setting**

PATHS

In [ ]:
# Path for storage
dtset_dir_parquet = Path('/content/drive/MyDrive/Data Lake IONClinics/S2ORC completo/spark')

In [ ]:
# Periodic Updates Reportings
path_historial = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/Historial.xlsx')
path_updates = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/Tabla_de_actualizaciones.xlsx')

# Database to be updated
path_BBDD_excel = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/BBDD_actualizado_urls_mayo_2025.xlsx')

# Updates Release Standarized protocols
path_standarized_release = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/updates')

# Database Copy
path_BBDD_copy = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/DataFrames Modulo 2/df_BBDD_actualizado_copy')

# Database cleaned (Proceso 1 y Proceso 2)
path_BBDD_cleaned = Path('/content/drive/MyDrive/Data Lake IONClinics/Actualizaciones/DataFrames Modulo 2/df_BBDD_actualizado_cleaned')

# Database Label Encoder
path_BBDD_labeled = Path('/content/drive/MyDrive/Data Lake IONClinics/DataBase/Protocolos/BBDD_label_encoder_mayo_2025.xlsx')

# Database Mapeo
path_BBDD_mapped = Path('/content/drive/MyDrive/Data Lake IONClinics/DataBase/Protocolos/BBDD_mapeado_mayo_2025.xlsx')

# Población Sintética
path_poblacion_sintetica = Path('/content/drive/MyDrive/Data Lake IONClinics/DataBase/Poblaciones/poblacion_sesgada_completa.xlsx')

### **Spark Session**

In [ ]:
from pyspark.sql import SparkSession
import findspark

# Initialize Spark
findspark.init()

# Create or get the SparkSession with custom configurations
spark = SparkSession.builder \
    .appName("tDCS_recomendador") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Create an RDD with increased partitions
rdd = spark.sparkContext.parallelize(range(100), numSlices=8)  # Increase the number of partitions

# Get the number of partitions
num_partitions = rdd.getNumPartitions()
print("Number of partitions (indirect indicator of workers):", num_partitions)


Number of partitions (indirect indicator of workers): 8


In [ ]:
spark

In [ ]:
dtset_dir_parquet = Path(dtset_dir_parquet)
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
hdfs_dir_parquet = spark._jvm.org.apache.hadoop.fs.Path(dtset_dir_parquet.as_posix())
if not fs.exists(hdfs_dir_parquet):
    fs.mkdirs(hdfs_dir_parquet)

## **PROCESO 1: Actualización de protocolos**

En este proceso se incorporan, modifican o eliminan protocolos dentro de la tabla correspondiente, permitiendo mantener actualizada la base de datos con la información más reciente y relevante para el sistema de recomendación.

In [ ]:
def initializer(path_historial, path_updates, path_BBDD_excel):
  """
  Initializes the historical, updates and deletes tables, and database. Initializes empty dictionaries to handle deletion and update information.
  """

  historial_1 = pd.read_excel(path_historial)

  updates_table_1 = pd.read_excel(path_updates)

  BBDD_1 = pd.read_excel(path_BBDD_excel)


  # Update dictionary -- restarts with each release (to put in history)
  updates_dict_init = {
      'corpusid': [],
      'DOI': [],
      'Title': [],
      'Evidence anterior': [],
      'Evidence_level anterior': [],
      'Evidence nuevo': [],
      'Evidence_level nuevo': []
  }
  # Dictionary with the information of each new update -- restarts with each new (temporary) update.
  new_update_dict_init = {
      'release': [],
      'corpusid': [],
      'DOI': [],
      'Title': [],
      'Evidence anterior': [],
      'Evidence_level anterior': [],
      'Evidence nuevo': [],
      'Evidence_level nuevo': []
  }

  return historial_1, updates_table_1, BBDD_1, updates_dict_init, new_update_dict_init

In [ ]:
def periodic_updates(path_historial, path_updates, path_BBDD_excel, path_standarized_release, path_BBDD_original):
  """
  Updates the protocol database according to the information in the historical, updates and deletes tables.
  """

  # Inicialization
  historial, updates_table, BBDD, updates_dict_init, new_update_dict_init = initializer(path_historial, path_updates, path_BBDD_excel)

  # UPDATES ------------------------------------------------------------

  # Create a dataframe of the protocols to be added as new ones to the database.
  df_to_update_new = pd.DataFrame(columns=BBDD.columns)

  for index, row in updates_table.iterrows():
    corpusid = row['corpusid']
    print(corpusid)
    last_release = row['release']

    evidence_anterior = row['Evidence anterior']
    evidence_nuevo = row['Evidence nuevo']
    valid_evidences = ['A', 'B', 'C', 'D']

    if evidence_anterior not in valid_evidences:
      evidence_anterior = 'o'
    if evidence_nuevo not in valid_evidences:
      evidence_nuevo = 'o'

    # If the protocol is new
    if row['Evidence anterior'] == '-':
      print('Hay un protocolo nuevo que añadir')
      path_standarized = path_standarized_release.joinpath(f'{last_release}/df_standarized_{last_release}')
      df_standarized = pd.read_csv(path_standarized)

      for index, row in df_standarized.iterrows():
        if row['corpusid'] == corpusid:
          print(row)
          fila_protocolo = df_standarized.iloc[[index]]
          df_to_update_new = pd.concat([df_to_update_new, fila_protocolo], ignore_index= True)

    # If the protocol already exists but with other (always greater) evidence -- the evidence is updated only
    else:
      # Establish an order of evidence and replace the 'Evidence' with its order value.
      evidence_order = {'A': 4, 'B': 3, 'C': 2, 'D': 1, 'o':0}
      evidence_anterior_order = [evidence_order[c] for c in evidence_anterior]
      evidence_nuevo_order = [evidence_order[c] for c in evidence_nuevo]

      # If you have further evidence -- update
      if evidence_nuevo_order > evidence_anterior_order:
        row_protocol_BBDD = BBDD.loc[BBDD['corpusid'] == corpusid]
        row_protocol_BBDD['Evidence'] = row['Evidence nuevo']
        row_protocol_BBDD['Evidence_level'] = row['Evidence_level nuevo']
        BBDD.update(row_protocol_BBDD)

      # If you have the same evidence -- look at evidence levels
      if evidence_nuevo_order == evidence_anterior_order:
        # If it has a higher level -- the level is updated.
        if row['Evidence_level nuevo'] > row['Evidence_level anterior']:
          row_protocol_BBDD = BBDD.loc[BBDD['corpusid'] == corpusid]
          row_protocol_BBDD['Evidence_level'] = row['Evidence_level nuevo']
          BBDD.update(row_protocol_BBDD)

      else:
        continue

  # Add this new dataframe (new protocols) to the database
  BBDD = pd.concat([BBDD, df_to_update_new], ignore_index= True)

  # Restart the update table (restart excel as well).
  updates_table = pd.DataFrame(columns = ['release', 'corpusid', 'Evidence anterior', 'Evidence_level anterior', 'Evidence nuevo', 'Evidence_level nuevo'])
  output_excel_path_2 = path_updates
  updates_table.to_excel(output_excel_path_2, index=False)

  # SAVE DATABASE -------------------------------------------------
  BBDD = BBDD.reset_index(drop=True)
  output_excel_path_BBDD = path_BBDD_excel
  BBDD.to_excel(output_excel_path_BBDD, index=False)
  output_excel_path_BBDD_2 = path_BBDD
  BBDD.to_csv(output_excel_path_BBDD_2, index=False)

In [ ]:
periodic_updates(path_historial, path_updates, path_BBDD_excel, path_standarized_release, path_BBDD)

247891069
Hay un protocolo nuevo que añadir
Unnamed: 0                                                               0
corpusid                                                         247891069
Year                                                                  2022
Title                    Effects of Transcranial Direct Current Stimula...
DOI                                               10.3390/brainsci12040452
Current (mA)                                                           0.0
Duration (min)                                                        20.0
Periodicity_info         20 minutes each session, intervals between 2 p...
Sessions                                                               0.0
Times_per_day                                                          0.0
Days_per_week                                                          0.0
Weeks                                                                  0.0
Cathode                                                 

<ipython-input-11-64b217e00e12>:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_to_update_new = pd.concat([df_to_update_new, fila_protocolo], ignore_index= True)


Unnamed: 0                                                              15
corpusid                                                         258672617
Year                                                                  2023
Title                    Exploring the Potential of Transcranial Direct...
DOI                                                   10.3390/life13051172
Current (mA)                                                           0.0
Duration (min)                                                        20.0
Periodicity_info                    20 min, five times weekly, for 2 weeks
Sessions                                                              10.0
Times_per_day                                                          1.0
Days_per_week                                                          5.0
Weeks                                                                  2.0
Cathode                                                      Not specified
Cathode_standarized      

## **PROCESO 2: Limpieza de registros**

Este proceso se encarga de depurar la base de datos de protocolos, eliminando aquellos registros que no cumplen con los criterios de inclusión. En particular, se descartan los siguientes casos:

1. Protocolos provenientes de **revisiones** (revisiones narrativas, revisiones sistemáticas, etc.).

2. Protocolos que consisten en **una única sesión**.

3. Protocolos que han demostrado ser **inefectivos**.

4. Protocolos cuyos **datos no pudieron ser extraídos completamente** mediante el uso del prompt de GPT.

In [ ]:
# Función personalizada que convierte a float si es posible, de lo contrario deja el valor original
def safe_float_conversion(val):
    try:
        return float(val)
    except ValueError:
        return val  # Si no se puede convertir, retorna el valor original

In [ ]:
def periodic_cleaning(path_BBDD_excel, path_BBDD_copy, path_BBDD_cleaned):
  """
  Periodic cleaning of the protocol database.
  """

  input_parquet_path = path_BBDD_excel
  df_DataBase = pd.read_excel(input_parquet_path)
  df_DataBase = df_DataBase.reset_index()

  if 'Unnamed: 0' in df_DataBase.columns:
    df_DataBase = df_DataBase.drop(columns=['Unnamed: 0'])
  if 'index' in df_DataBase.columns:
    df_DataBase = df_DataBase.drop(columns=['index'])
  if 'Unnamed: 0.1' in df_DataBase.columns:
    df_DataBase = df_DataBase.drop(columns=['Unnamed: 0.1'])

  # We create a copy of the database to be cleaned -- we save the copy
  df_cleaned = df_DataBase.copy()
  path_save = path_BBDD_copy
  df_cleaned.to_csv(path_save)
  print(df_cleaned)

  df_cleaned['Current (mA)'] = df_cleaned['Current (mA)'].apply(safe_float_conversion)
  df_cleaned['Duration (min)'] = df_cleaned['Duration (min)'].apply(safe_float_conversion)
  df_cleaned['Sessions'] = df_cleaned['Sessions'].apply(safe_float_conversion)
  df_cleaned['Times_per_day'] = df_cleaned['Times_per_day'].apply(safe_float_conversion)
  df_cleaned['Days_per_week'] = df_cleaned['Days_per_week'].apply(safe_float_conversion)
  df_cleaned['Weeks'] = df_cleaned['Weeks'].apply(safe_float_conversion)


  for index, row in df_cleaned.iterrows():

    # Delete all records that are reviews -- as the information is usually not well extracted.
    title = row['Title']
    if 'review' in title.lower():
      df_cleaned.drop(index, inplace=True)
      continue

    # Delete all records of protocols that have only a single session -- they are not protocols.
    sessions = row['Sessions']
    if sessions == 1:
      df_cleaned.drop(index, inplace=True)
      continue

    # Eliminate those that are NOT effective -- they will never be recommended.
    effectiveness = row['Effectiveness']
    if effectiveness == 'NO':
      df_cleaned.drop(index, inplace=True)
      continue

    # Eliminate those with 'Not Specified' in any of the columns: 'Current (mA)', 'Duration (min)', 'Periodicity_info', 'Cathode', 'Anode', 'Pathology', 'Evidence'.
    current = row['Current (mA)']
    duration = row['Duration (min)']
    sessions = row['Sessions']
    periodicity_info = row['Periodicity_info']
    cathode = row['Cathode']
    anode = row['Anode']
    pathology = row['Pathology']
    evidence = row['Evidence']
    if current == 0 or duration == 0 or sessions == 0 or periodicity_info == 'Not specified' or periodicity_info == 'not specified' or cathode == 'Not specified' or anode == 'Not specified' or pathology == 'Not specified' or evidence == 'Not specified':
      df_cleaned.drop(index, inplace=True)
      continue

  # Save the cleaned database
  df_cleaned = df_cleaned.reset_index(drop=True)

  if 'Unnamed: 0' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=['Unnamed: 0'])
  if 'index' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=['index'])
  if 'Unnamed: 0.1' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=['Unnamed: 0.1'])

  # Eliminate duplicates -- keep highest evidence
  df_cleaned['original_index'] = df_cleaned.index
  evidence_order = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
  df_cleaned['evidence_rank'] = df_cleaned['Evidence'].map(lambda x: evidence_order.get(x, 5))

  df_sorted = df_cleaned.sort_values(by=['evidence_rank', 'Evidence_level'], ascending=[True, False])
  duplicados = df_sorted[df_sorted.duplicated(subset='corpusid', keep=False)]
  grupos_duplicados = duplicados.groupby('corpusid')
  for corpusid, group in grupos_duplicados:
      print(f"\nDuplicados encontrados para corpusid '{corpusid}':")
      print(group[['corpusid', 'Evidence', 'Evidence_level', 'original_index']])

  df_deduplicated = df_sorted.drop_duplicates(subset='corpusid', keep='first')
  df_final = df_deduplicated.sort_values(by='original_index').drop(columns=['original_index', 'evidence_rank'])

  df_final = df_final.reset_index(drop=True)
  path_save = path_BBDD_cleaned
  df_final.to_csv(path_save)

  return df_final

In [ ]:
df_DataBase_cleaned = periodic_cleaning(path_BBDD_excel, path_BBDD_copy, path_BBDD_cleaned)
df_DataBase_cleaned

      corpusid  Year                                              Title  \
0    251538675  2022  The effect of transcranial direct current stim...   
1    271063936  2024  Effect of transcranial direct current stimulat...   
2    237796706  2021  The Effect of Medication Therapy Combined with...   
3    273786214  2024  ANODAL TDCS AND VIRTUAL REALITY GAIT REHABILIT...   
4    243943989  2021  Limited Add-On Effects of Unilateral and Bilat...   
..         ...   ...                                                ...   
507  235816557  2023  Transcranial Direct Current Stimulation Revers...   
508  247750301  2022  The Advantages of Non-Invasive Brain Stimulati...   
509  231874040  2021  The Effect of Non-Invasive Brain Stimulation (...   
510  218908899  2020  Transcranial Direct Current Stimulation to Fac...   
511  211024198  2020  Patient-Controlled Intravenous Morphine Analge...   

                                            DOI Current (mA)  Duration (min)  \
0                  

,corpusid,Year,Title,DOI,Current (mA),Duration (min),Periodicity_info,Sessions,Times_per_day,Days_per_week,...,Age,Kid_0_5,Youth_6_17,Adult_18_31,Adult_32_59,Elderly_60_100,Origin,Origin_standarized,url,Razones de eliminacion
0,251538675,2022,The effect of transcranial direct current stim...,10.5114/areh.2022.116529,2.0,20.0,2ma for 20 min for 5 consecutive days,5.0,1.0,5.0,...,16-29,0,1,1,0,0,India,Asia,https://www.termedia.pl/Journal/-125/pdf-47097...,NaN
1,271063936,2024,Effect of transcranial direct current stimulat...,10.1038/s41398-024-02994-w,2.0,20.0,2 sessions of tdcs during n-back task were del...,10.0,2.0,5.0,...,24-42,0,0,1,1,0,Singapore,Asia,https://www.nature.com/articles/s41398-024-029...,NaN
2,237796706,2021,The Effect of Medication Therapy Combined with...,https://doi.org/10.21203/rs.3.rs-519087/v1,2.0,20.0,"10 days, 2 sessions per day each for 20 min",20.0,2.0,0.0,...,21-42,0,0,1,1,0,Iran,Asia,https://www.researchsquare.com/article/rs-5190...,NaN
3,273786214,2024,ANODAL TDCS AND VIRTUAL REALITY GAIT REHABILIT...,https://digitalcommons.uri.edu/theses/2471,2.0,30.0,ten 30-minute sessions over 2 weeks,10.0,1.0,5.0,...,18-75,0,0,1,1,1,USA,America,https://digitalcommons.uri.edu/cgi/viewcontent...,NaN
4,243943989,2021,Limited Add-On Effects of Unilateral and Bilat...,10.3389/fneur.2021.736075,1.0,23.0,5 days of visuo-motor grip force tracking task...,5.0,1.0,5.0,...,45-74,0,0,0,1,1,Germany,Europe,https://www.frontiersin.org/articles/10.3389/f...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,227164756,2020,Transcranial direct current stimulation (tDCS)...,Not specified,2.0,30.0,five consecutive sessions for 2 weeks,10.0,1.0,5.0,...,45-60,0,0,0,1,1,Egypt,Africa,https://ejnpn.springeropen.com/track/pdf/10.11...,NaN
249,234351703,2022,Clinical study on swallowing function of brain...,10.1007/s10072-021-05247-6,1.6,20.0,"once a day, 6days each week, for a total of 8 ...",48.0,1.0,6.0,...,60-70,0,0,0,0,1,China,Asia,https://link.springer.com/content/pdf/10.1007/...,NaN
250,201765483,2019,Timing-dependent interaction effects of transc...,10.1016/j.jns.2019.116436,1.0,30.0,five days per week for 2 weeks,10.0,1.0,5.0,...,NaN,0,0,0,0,0,Not specified,Not specified,http://ira.lib.polyu.edu.hk/bitstream/10397/97...,NaN
251,221540862,2020,Functional near-infrared spectroscopy to asses...,10.1111/jon.12782,2.0,20.0,10-day active tdcs and mbm regimen over two we...,10.0,1.0,5.0,...,50-85,0,0,0,1,1,USA,America,https://europepmc.org/articles/pmc7719610?pdf=...,NaN


## **PROCESO 3: Codificación de etiquetas (Label Encoder) y Mapeo de variables categóricas**

En esta etapa se asigna un identificador único a cada combinación específica de parámetros que definen un protocolo. Esta codificación facilita la gestión, comparación y análisis de los protocolos dentro del sistema, permitiendo su uso eficiente en modelos de recomendación y procesos automatizados de selección. Posterioremente, se realiza la conversión de los valores categóricos a valores numéricos, con el objetivo de preparar los datos para su procesamiento por parte del modelo de recomendación. Este mapeo es esencial para garantizar la compatibilidad con los algoritmos de aprendizaje automático, los cuales requieren entradas numéricas para su correcto funcionamiento.

In [ ]:
df_DataBase_cleaned = pd.read_csv(path_BBDD_cleaned)

In [ ]:
from sklearn.preprocessing import LabelEncoder

def process_tdcs_dataframe(df_DataBase_cleaned):
    """
    Combines protocol label creation and numerical feature mapping for tDCS data.

    Parameters:
        df_DataBase_cleaned (pd.DataFrame): Raw dataframe with original protocol and demographic info.

    Returns:
        df_DataBase_cleaned (pd.DataFrame): Original dataframe with added 'Protocol_Label'.
        df_mapped (pd.DataFrame): Numerically mapped dataframe ready for model training.
        le (LabelEncoder): Trained label encoder for decoding protocol labels.
    """

    # ---------------- Protocol Label Encoding ---------------- #
    protocol_columns = ['Current (mA)', 'Duration (min)', 'Times_per_day',
                        'Days_per_week', 'Cathode_standarized', 'Anode_standarized', 'Modality']

    df_DataBase_cleaned['Protocol_Identifier'] = df_DataBase_cleaned[protocol_columns] \
        .apply(lambda row: '_'.join(row.values.astype(str)), axis=1)

    le = LabelEncoder()
    df_DataBase_cleaned['Protocol_Label'] = le.fit_transform(df_DataBase_cleaned['Protocol_Identifier'])

    df_DataBase_cleaned.drop(columns=['Protocol_Identifier'], inplace=True)

    df_cleaned_with_labels = df_DataBase_cleaned.copy()

    # ---------------- Numerical Mappings ---------------- #
    # Create mappings
    unique_values_current = df_DataBase_cleaned['Current (mA)'].unique()
    unique_values_current.sort()
    current_mapping = {current: idx + 1 for idx, current in enumerate(unique_values_current)}

    electrode_mapping = {
        'Not specified': 0, 'others': 1, 'C3/C4': 2, 'F3/F4': 3, 'Fp1/Fp2': 4,
        'F7/F8': 5, 'P3/P4': 6, 'Cz': 7, 'Supraorbital region': 8, 'T3/T4': 9,
        'O1/O2': 10, 'Fz': 11, 'FpZ': 12, 'Oz': 13, 'Anatomical area': 14
    }

    symptom_keyword_mapping = {
        'Not specified': 0, 'others': 1, 'motor': 2, 'depression': 3, 'chronic pain': 4,
        'aphasia': 5, 'complex symptoms associated with fibromyalgia': 6, 'cognitive function': 7,
        'neuropathic pain': 8, 'pain associated with fibromyalgia': 9,
        'complex symptoms associated with schizophrenia': 10, 'memory': 11
    }

    pathology_keyword_mapping = {
        'Not specified': 0, 'others': 1, 'stroke': 2, 'pain': 3, 'depression': 4,
        'schizophrenia': 5, 'spinal cord injury': 6, 'fibromyalgia': 7,
        'cognitive decline': 8, 'knee osteoarthritis': 9, 'mental-health disorder': 10,
        'cancer': 11
    }

    modality_mapping = {
        'Not specified': 0, 'others': 1, 'Anodal': 2, 'Bilateral': 3,
        'High-definition': 4, 'Cathodal': 5, 'Multifocal': 6, 'TMS': 7,
        'tDCS combined': 8
    }

    evidence_mapping = {'A': 4, 'B': 3, 'C': 2, 'D': 1}

    # Map columns numerically
    df_DataBase_cleaned['Current_map'] = df_DataBase_cleaned['Current (mA)'].map(current_mapping).fillna(-1).astype(int)
    df_DataBase_cleaned['Cathode_standarized_map'] = df_DataBase_cleaned['Cathode_standarized'].map(electrode_mapping).fillna(-1).astype(int)
    df_DataBase_cleaned['Anode_standarized_map'] = df_DataBase_cleaned['Anode_standarized'].map(electrode_mapping).fillna(-1).astype(int)
    df_DataBase_cleaned['Pathology_map'] = df_DataBase_cleaned['Pathology_standarized'].map(pathology_keyword_mapping).fillna(-1).astype(int)
    df_DataBase_cleaned['Symptom_map'] = df_DataBase_cleaned['Symptom_Keyword'].map(symptom_keyword_mapping).fillna(-1).astype(int)
    df_DataBase_cleaned['Modality_map'] = df_DataBase_cleaned['Modality'].map(modality_mapping).fillna(-1).astype(int)
    df_DataBase_cleaned['Evidence'] = df_DataBase_cleaned['Evidence'].map(evidence_mapping).fillna(-1).astype(int)

    df_DataBase_cleaned["Adult_18_59"] = df_DataBase_cleaned.apply(
        lambda row: 1 if row["Adult_18_31"] == 1 or row["Adult_32_59"] == 1 else 0, axis=1)

    # Drop merged columns
    df_DataBase_cleaned.drop(columns=["Adult_18_31", "Adult_32_59"], inplace=True)

    # Select final mapped dataframe
    df_mapped = df_DataBase_cleaned[[
        'Current_map', 'Duration (min)', 'Times_per_day', 'Days_per_week', 'Weeks',
        'Cathode_standarized_map', 'Anode_standarized_map', 'Modality_map',
        'Pathology_map', 'Symptom_map', 'Evidence', 'Evidence_level', 'N_sample',
        'Protocol_Label', 'Males', 'Females', 'No_gender', 'Age',
        'Kid_0_5', 'Youth_6_17', 'Adult_18_59', 'Elderly_60_100', 'Origin_standarized'
    ]].copy()

    return df_cleaned_with_labels, df_mapped


In [ ]:
df_cleaned_with_labels, df_mapped_numerical = process_tdcs_dataframe(df_DataBase_cleaned)
df_cleaned_with_labels.head(20)

,Unnamed: 0,corpusid,Year,Title,DOI,Current (mA),Duration (min),Periodicity_info,Sessions,Times_per_day,...,Kid_0_5,Youth_6_17,Adult_18_31,Adult_32_59,Elderly_60_100,Origin,Origin_standarized,url,Razones de eliminacion,Protocol_Label
0,0,251538675,2022,The effect of transcranial direct current stim...,10.5114/areh.2022.116529,2.0,20.0,2ma for 20 min for 5 consecutive days,5.0,1.0,...,0,1,1,0,0,India,Asia,https://www.termedia.pl/Journal/-125/pdf-47097...,NaN,88
1,1,271063936,2024,Effect of transcranial direct current stimulat...,10.1038/s41398-024-02994-w,2.0,20.0,2 sessions of tdcs during n-back task were del...,10.0,2.0,...,0,0,1,1,0,Singapore,Asia,https://www.nature.com/articles/s41398-024-029...,NaN,128
2,2,237796706,2021,The Effect of Medication Therapy Combined with...,https://doi.org/10.21203/rs.3.rs-519087/v1,2.0,20.0,"10 days, 2 sessions per day each for 20 min",20.0,2.0,...,0,0,1,1,0,Iran,Asia,https://www.researchsquare.com/article/rs-5190...,NaN,118
3,3,273786214,2024,ANODAL TDCS AND VIRTUAL REALITY GAIT REHABILIT...,https://digitalcommons.uri.edu/theses/2471,2.0,30.0,ten 30-minute sessions over 2 weeks,10.0,1.0,...,0,0,1,1,1,USA,America,https://digitalcommons.uri.edu/cgi/viewcontent...,NaN,157
4,4,243943989,2021,Limited Add-On Effects of Unilateral and Bilat...,10.3389/fneur.2021.736075,1.0,23.0,5 days of visuo-motor grip force tracking task...,5.0,1.0,...,0,0,0,1,1,Germany,Europe,https://www.frontiersin.org/articles/10.3389/f...,NaN,27
5,5,214763880,2020,The Effect of Transcranial Direct Current Stim...,10.3389/fphar.2020.00401,2.0,20.0,7 sessions in 2 consecutive weeks,7.0,1.0,...,0,0,1,1,0,Iran,Asia,https://www.frontiersin.org/articles/10.3389/f...,NaN,90
6,6,235691272,2021,Transcranial electrostimulation with special w...,10.1186/s12984-021-00901-8,1.5,20.0,"18 treatment sessions of 1 hour each, 3 days a...",18.0,1.0,...,0,0,0,1,1,Taiwan,Not specified,https://jneuroengrehab.biomedcentral.com/track...,NaN,40
7,7,261138401,2023,Home-administered transcranial direct current ...,10.3389/fpsyt.2023.1199773,2.0,30.0,"28 sessions (5 sessions/week, 6 weeks) followe...",32.0,1.0,...,0,0,1,1,1,United States,America,https://www.frontiersin.org/articles/10.3389/f...,NaN,152
8,8,253841801,2022,Short term effects of anodal cerebellar vs. an...,10.3389/fnins.2022.1035558,2.0,20.0,3 sessions of anodal transcranial direct curre...,3.0,1.0,...,0,0,0,1,1,China,Asia,https://www.frontiersin.org/articles/10.3389/f...,NaN,77
9,9,256909742,2017,Effects of transcranial direct current stimula...,10.1038/s41598-017-03173-2,2.0,15.0,15 minutes 2 ma anodal tdcs over left m1 and p...,2.0,1.0,...,0,0,1,1,0,Spain,Europe,https://www.nature.com/articles/s41598-017-031...,NaN,60


In [ ]:
## Save Labeled DDBB
df_cleaned_with_labels.to_excel(path_BBDD_labeled, index=False)
print(f"DataFrame saved at: {path_BBDD_labeled}")

#Save Mapped DDBB
df_mapped_numerical.to_excel(path_BBDD_mapped, index=False)
print(f"DataFrame saved at: {path_BBDD_mapped}")

DataFrame saved at: /content/drive/MyDrive/ALTERNATIVAS_SEMANTIC_SCHOLAR/DataFrames Modulo 2/BBDD_label_encoder_mayo_2025.xlsx
DataFrame saved at: /content/drive/MyDrive/ALTERNATIVAS_SEMANTIC_SCHOLAR/DataFrames Modulo 2/BBDD_mapeado_mayo_2025.xlsx


## **PROCESO 4: Diseño de población sintética**

En esta etapa se genera una población sintética de pacientes, creada de forma aleatoria pero asegurando la cobertura de todas las combinaciones posibles de sintomatología y patología presentes en la base de datos de protocolos. Esta población artificial permite entrenar los modelos recomendadores bajo distintos escenarios clínicos, facilitando la evaluación y validación de su desempeño antes de su aplicación en casos reales.

In [ ]:
def generate_synthetic_users_with_protocols(df_cleaned_with_labels, n_samples=300, num_sessions=10):
    """
    Generates synthetic patients and assigns unique valid and exact protocols
    based on symptom and pathology matching.

    Args:
        df_cleaned_with_labels (pd.DataFrame): Cleaned protocol dataframe with 'Symptom_Keyword',
                                               'Pathology_standarized', and 'Protocol_Label'.
        n_samples (int): Total number of synthetic users to generate.
        num_sessions (int): Number of therapy sessions per response profile.

    Returns:
        pd.DataFrame: Synthetic patients dataframe with 'Protocols' and 'Exact_protocols' columns.
    """

    # Define distributions and mappings
    age_distribution = np.concatenate([
        np.random.normal(loc=10, scale=3, size=int(n_samples * 0.25)),
        np.random.normal(loc=40, scale=10, size=int(n_samples * 0.5)),
        np.random.normal(loc=80, scale=10, size=int(n_samples * 0.25))
    ])
    age_distribution = np.clip(age_distribution, 6, 100).astype(int)

    age_restrictions = {
        'stroke': (18, 100), 'pain': (6, 100), 'fibromyalgia': (18, 100), 'schizophrenia': (18, 100),
        'depression': (6, 100), 'knee osteoarthritis': (32, 100), 'spinal cord injury': (0, 100),
        'mental-health disorder': (6, 100), 'cognitive decline': (32, 100), 'cancer': (0, 100)
    }

    symptoms_by_pathology = {
        'stroke': ['Motor', 'Aphasia', 'Cognitive function', 'Depression'],
        'pain': ['Chronic pain', 'Neuropathic pain', 'Depression'],
        'fibromyalgia': ['Chronic pain', 'Pain associated with fibromyalgia', 'Complex symptoms associated with fibromyalgia', 'Depression'],
        'schizophrenia': ['Depression', 'Cognitive function', 'Complex symptoms associated with schizophrenia'],
        'depression': ['Depression', 'Cognitive function'],
        'knee osteoarthritis': ['Chronic pain', 'Motor'],
        'spinal cord injury': ['Motor', 'Neuropathic pain', 'Chronic pain', 'Depression'],
        'mental-health disorder': ['Depression', 'Cognitive function', 'Complex symptoms associated with schizophrenia'],
        'cognitive decline': ['Cognitive function', 'Depression'],
        'cancer': ['Chronic pain', 'Neuropathic pain', 'Depression', 'Cognitive function']
    }

    def generate_response_profile(profile_type):
        if profile_type == 1:
            return [random.choice([4, 5]) for _ in range(num_sessions)]
        elif profile_type == 2:
            return [random.choice([4, 5]) if i < num_sessions // 2 else random.choice([1, 2]) for i in range(num_sessions)]
        elif profile_type == 3:
            return [random.choice([1, 2]) if i < num_sessions // 2 else random.choice([4, 5]) for i in range(num_sessions)]
        elif profile_type == 4:
            return [random.choice([1, 2]) for _ in range(num_sessions)]

    # Generate synthetic users
    users = []
    genders = ['Male', 'Female']
    age_groups = ['Youth_6_17', 'Adult_18_59', 'Elderly_60_100']

    for pathology, symptoms in symptoms_by_pathology.items():
        age_min, age_max = age_restrictions[pathology]

        for gender in genders:
            for age_group in age_groups:
                valid_ages = age_distribution[(age_distribution >= age_min) & (age_distribution <= age_max)]
                if age_group == 'Youth_6_17':
                    age_range = valid_ages[(valid_ages >= 6) & (valid_ages <= 17)]
                elif age_group == 'Adult_18_59':
                    age_range = valid_ages[(valid_ages >= 18) & (valid_ages <= 59)]
                elif age_group == 'Elderly_60_100':
                    age_range = valid_ages[(valid_ages >= 60) & (valid_ages <= 100)]
                if len(age_range) == 0:
                    continue

                for profile_type in range(1, 5):
                    for _ in range(len(age_range) // 10):
                        age = random.choice(age_range)
                        symptom = random.choice(symptoms)
                        response_profile = generate_response_profile(profile_type)
                        user = {
                            'Gender': gender,
                            'Age': age,
                            'Age_group': age_group,
                            'Pathology': pathology,
                            'Symptom': symptom,
                            'Response_profile': response_profile
                        }
                        users.append(user)

    synthetic_users_df = pd.DataFrame(users)

    # Assign unique valid and exact protocols
    synthetic_users_df['Protocols'] = [[] for _ in range(len(synthetic_users_df))]
    synthetic_users_df['Exact_protocols'] = [[] for _ in range(len(synthetic_users_df))]

    for idx, row in synthetic_users_df.iterrows():
        valid_protocols = set()
        exact_protocols = set()

        symptom = row['Symptom'].lower()
        pathology = row['Pathology'].lower()

        df_filtered = df_cleaned_with_labels[
            df_cleaned_with_labels['Symptom_Keyword'].str.lower() == symptom
        ]

        for _, protocol_row in df_filtered.iterrows():
            corpusid = protocol_row['Protocol_Label']
            valid_protocols.add(corpusid)
            if protocol_row['Pathology_standarized'].lower() == pathology:
                exact_protocols.add(corpusid)

        synthetic_users_df.at[idx, 'Protocols'] = sorted(list(valid_protocols))
        synthetic_users_df.at[idx, 'Exact_protocols'] = sorted(list(exact_protocols))

    return synthetic_users_df


In [ ]:
synthetic_users_df = generate_synthetic_users_with_protocols(df_cleaned_with_labels)
synthetic_users_df.head(10)

,Gender,Age,Age_group,Pathology,Symptom,Response_profile,Protocols,Exact_protocols
0,Male,37,Adult_18_59,stroke,Motor,"[5, 5, 5, 5, 5, 4, 5, 4, 4, 4]","[0, 1, 2, 3, 5, 7, 9, 10, 12, 14, 15, 16, 19, ...","[0, 2, 3, 5, 7, 9, 12, 14, 15, 16, 19, 20, 25,..."
1,Male,48,Adult_18_59,stroke,Aphasia,"[4, 4, 5, 5, 5, 4, 4, 4, 5, 5]","[6, 13, 21, 23, 24, 51, 64, 66, 91, 93, 94, 96]","[6, 13, 21, 23, 24, 51, 64, 66, 91, 93, 94, 96]"
2,Male,43,Adult_18_59,stroke,Depression,"[5, 4, 5, 5, 4, 4, 4, 4, 4, 4]","[11, 18, 22, 34, 38, 47, 50, 73, 75, 90, 91, 9...","[146, 156]"
3,Male,30,Adult_18_59,stroke,Aphasia,"[4, 4, 4, 5, 5, 4, 4, 5, 4, 4]","[6, 13, 21, 23, 24, 51, 64, 66, 91, 93, 94, 96]","[6, 13, 21, 23, 24, 51, 64, 66, 91, 93, 94, 96]"
4,Male,33,Adult_18_59,stroke,Cognitive function,"[5, 4, 5, 4, 5, 5, 4, 5, 4, 5]","[17, 20, 37, 69, 70, 90, 100, 102, 106, 114, 1...","[17, 20, 70, 100, 149, 150, 158]"
5,Male,31,Adult_18_59,stroke,Depression,"[5, 4, 4, 5, 5, 5, 5, 5, 5, 5]","[11, 18, 22, 34, 38, 47, 50, 73, 75, 90, 91, 9...","[146, 156]"
6,Male,42,Adult_18_59,stroke,Cognitive function,"[5, 5, 4, 4, 4, 5, 4, 4, 5, 4]","[17, 20, 37, 69, 70, 90, 100, 102, 106, 114, 1...","[17, 20, 70, 100, 149, 150, 158]"
7,Male,30,Adult_18_59,stroke,Motor,"[5, 4, 4, 4, 5, 4, 4, 4, 4, 4]","[0, 1, 2, 3, 5, 7, 9, 10, 12, 14, 15, 16, 19, ...","[0, 2, 3, 5, 7, 9, 12, 14, 15, 16, 19, 20, 25,..."
8,Male,26,Adult_18_59,stroke,Cognitive function,"[4, 5, 4, 5, 4, 5, 4, 5, 4, 5]","[17, 20, 37, 69, 70, 90, 100, 102, 106, 114, 1...","[17, 20, 70, 100, 149, 150, 158]"
9,Male,35,Adult_18_59,stroke,Motor,"[5, 4, 5, 5, 5, 4, 5, 5, 5, 4]","[0, 1, 2, 3, 5, 7, 9, 10, 12, 14, 15, 16, 19, ...","[0, 2, 3, 5, 7, 9, 12, 14, 15, 16, 19, 20, 25,..."


In [ ]:
synthetic_users_df.to_excel(path_poblacion_sintetica, index=False)
print(f"DataFrame saved at: {path_poblacion_sintetica}")

DataFrame saved at: /content/drive/MyDrive/ALTERNATIVAS_SEMANTIC_SCHOLAR/DataFrames Modulo 2/poblacion_sesgada_completa.xlsx
